# 🧑‍🤝‍🧑 Gender Detection — End-to-End Deep Learning Project
**Project:** PRAICP-1001-GenderDetc · **Business case:** a machine-learning model that predicts the **gender (Male / Female)** of the person in an image.

**Project goals (from the project brief)**
1. **Image preprocessing**
2. **ML model to predict the gender**

**Workflow:** data audit & EDA → preprocessing (resize, normalise, class-aware augmentation) → four models (Dense baseline, CNN v1, CNN v2, MobileNetV2 transfer learning) → comparison → error analysis → prediction on a new image → save & reload.

> **How to run:** use **Kernel → Restart & Run All**. The cells depend on each other and must run in order (the previous version of this notebook failed because it was run out of order after a kernel restart). Everything configurable lives in the *Setup* cell.

**What is new compared with the previous version**
- Single *Setup* cell + model-builder functions, so a cell can never fail on a missing variable after a restart.
- **Duplicate-image check** before splitting, so identical photos cannot sit in both train and test (data leakage).
- Images are resized to **128 × 128** instead of 224 × 224: the source images are tiny, so 224 px adds compute (CNN v2 took ~250 s/epoch) but no information.
- **Fresh metric objects per model** (sharing one `AUC` object across models is a bug) and training accuracy measured on **un-augmented** training images.
- Validation scores are reported at the **restored best weights**; the test accuracy comes with a **95 % confidence interval**; the saved model is reloaded and verified.

## 0. Setup

In [ ]:
import os, re, json, time, zipfile, hashlib, pathlib, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- configuration ----------------
DATA_URL = "https://d3ilbtxij3aepc.cloudfront.net/projects/AI-Capstone-Projects/PRAICP-1001-GenderDetc.zip"
ZIP_PATH = "PRAICP-1001-GenderDetc.zip"
path     = "PRAICP-1001-GenderDetc"          # extraction folder (or set to your own image folder)

IMG_SIZE   = 128                              # images are tiny; 128 px is enough and ~3x faster than 224
BATCH_SIZE = 32
EPOCHS     = dict(baseline=15, cnn1=25, cnn2=30, tl1=15, tl2=20)   # EarlyStopping stops sooner if val_loss stops improving
PRETRAINED_WEIGHTS = "imagenet"               # needs internet on first run; None = random init (not recommended)

print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

## 1. Download and inspect the dataset

In [ ]:
if not os.path.isdir(path):
    if not os.path.exists(ZIP_PATH):
        print("Downloading dataset ...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(path)
print("Dataset folder:", path)
print(os.listdir(path))

**Label every image from its folder name.** We do not assume a fixed layout (`Male/`, `Female/`, `train/male/`, …): the nearest parent folder whose name contains the *word* male/female (or man/woman…) gives the label. Names are split into words, so a folder such as `Documents` can never be mistaken for `men`, and `female` is tested before `male`.

In [ ]:
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
FEMALE_TOK = {"female", "females", "woman", "women"}
MALE_TOK   = {"male", "males", "man", "men"}

def label_from_path(p):
    for part in reversed(p.parts[:-1]):
        low = part.lower()
        toks = set(re.split(r"[^a-z]+", low))
        if toks & FEMALE_TOK or low.startswith("female"): return "female"
        if toks & MALE_TOK   or low.startswith("male"):   return "male"
    return None

rows = []
for p in pathlib.Path(path).rglob("*"):
    if p.suffix.lower() in IMG_EXTS and not p.name.startswith("."):
        lab = label_from_path(p)
        if lab is not None:
            rows.append([str(p), lab])

df = pd.DataFrame(rows, columns=["image_path", "label"])
print("Total labelled images found:", len(df))
assert len(df) > 0, "No images found - check `path` and the folder names."
df.head()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Sample images

In [ ]:
plt.figure(figsize=(14, 6)); i = 1
for cls in sorted(df["label"].unique()):
    for img_path in df[df["label"] == cls].sample(6, random_state=SEED)["image_path"]:
        plt.subplot(2, 6, i); plt.imshow(Image.open(img_path).convert("RGB")); plt.title(cls); plt.axis("off"); i += 1
plt.tight_layout(); plt.show()

### 2.2 Class balance

In [ ]:
class_counts = df["label"].value_counts().sort_index()
print(class_counts)
plt.figure(figsize=(6, 4))
bars = plt.bar(class_counts.index, class_counts.values, color=["#d6619a", "steelblue"][:len(class_counts)])
for b, v in zip(bars, class_counts.values):
    plt.text(b.get_x() + b.get_width() / 2, b.get_height(), str(v), ha="center", va="bottom")
plt.ylabel("Number of images"); plt.title("Class distribution"); plt.tight_layout(); plt.show()
print("Imbalance ratio (max/min):", round(class_counts.max() / class_counts.min(), 2))

### 💡 Insight
A ratio close to 1.0 means the classes are balanced, so plain accuracy is a fair metric. We still pass class weights to the optimiser (harmless when balanced, protective if the data changes).

### 2.3 Data quality: corrupt files, image sizes, colour modes

In [ ]:
corrupt, sizes, modes = [], [], []
for img_path in df["image_path"]:
    try:
        with Image.open(img_path) as im: im.verify()
        with Image.open(img_path) as im: sizes.append(im.size); modes.append(im.mode)
    except Exception as e:
        corrupt.append((img_path, str(e)))

print(f"Scanned: {len(sizes) + len(corrupt)} | corrupt/unreadable: {len(corrupt)}")
for f, err in corrupt[:10]: print(" -", f, "->", err)
w, h = np.array([s[0] for s in sizes]), np.array([s[1] for s in sizes])
print(f"Width  px: min {w.min()}, median {int(np.median(w))}, max {w.max()}")
print(f"Height px: min {h.min()}, median {int(np.median(h))}, max {h.max()}")
print("Colour modes:", dict(Counter(modes)))

if corrupt:
    bad = {f for f, _ in corrupt}
    df = df[~df["image_path"].isin(bad)].reset_index(drop=True)
    print("Removed corrupt files. Remaining:", len(df))

plt.figure(figsize=(6, 4)); plt.hist(np.maximum(w, h), bins=40, color="slateblue"); plt.xlabel("longest side (px)"); plt.title("Image size distribution"); plt.show()

### 💡 Insight
The images come in many different (and small) sizes, so **resizing is mandatory** and the resolution we choose matters little beyond the typical image size — that is why 128 × 128 is used here. Unreadable files are removed so they cannot crash training.

### 2.4 Duplicate-image check (prevents train/test leakage)

In [ ]:
def dhash(path_, size=16):
    # perceptual 'difference hash': identical / near-identical pictures give the same hash
    with Image.open(path_) as im:
        g = np.asarray(im.convert("L").resize((size + 1, size), Image.LANCZOS), dtype=np.int16)
    return hashlib.md5((g[:, 1:] > g[:, :-1]).tobytes()).hexdigest()

df["hash"] = df["image_path"].map(dhash)
grp = df.groupby("hash")
dup_groups = grp.filter(lambda g: len(g) > 1)
conflict = grp.filter(lambda g: g["label"].nunique() > 1)
print(f"Images that have at least one duplicate: {len(dup_groups)} (in {dup_groups['hash'].nunique()} groups)")
print(f"Duplicate groups with CONFLICTING labels: {conflict['hash'].nunique()}")

n0 = len(df)
df = df[~df["hash"].isin(conflict["hash"])]            # contradictory labels: drop entirely
df = df.drop_duplicates("hash", keep="first").drop(columns="hash").reset_index(drop=True)
print(f"Images before: {n0} -> after de-duplication: {len(df)}")
print(df["label"].value_counts().to_dict())

### 💡 Insight
If the same photo appears in both the training and the test set, the test score is inflated because the model has effectively seen the answer. De-duplicating **before** the split removes that risk. If the count above is 0, the dataset is clean in this respect.

### 2.5 Shuffle and split: Train / Validation / Test = 70 / 15 / 15 (stratified)

In [ ]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_data, temp_data = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_data, test_data   = train_test_split(temp_data, test_size=0.50, random_state=SEED, stratify=temp_data["label"])
train_data, val_data, test_data = [d.reset_index(drop=True) for d in (train_data, val_data, test_data)]
for n, d in [("Train", train_data), ("Validation", val_data), ("Test", test_data)]:
    print(f"{n:11s}: {len(d):5d}  {d['label'].value_counts().to_dict()}")

## 3. Preprocessing
- **Resize** every image to 128 × 128, **normalise** pixels to 0–1.
- **Augmentation on the training set only** (validation/test must stay untouched so their scores are trustworthy). A horizontal flip is safe for faces; rotation, shift, zoom and brightness changes are kept mild so the face stays recognisable.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.15, brightness_range=(0.8, 1.2), horizontal_flip=True, fill_mode="nearest")
plain_datagen = ImageDataGenerator(rescale=1.0 / 255)          # validation / test / un-augmented train: rescale only

def make_gen(datagen, frame, shuffle):
    return datagen.flow_from_dataframe(
        dataframe=frame, x_col="image_path", y_col="label", target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode="binary", shuffle=shuffle, seed=SEED)

train_generator      = make_gen(train_datagen, train_data, True)     # augmented - used for training
train_eval_generator = make_gen(plain_datagen, train_data, False)    # same images, no augmentation - used for fair evaluation
validation_generator = make_gen(plain_datagen, val_data,   False)
test_generator       = make_gen(plain_datagen, test_data,  False)

class_indices = train_generator.class_indices                       # {'female': 0, 'male': 1}
class_labels  = [k for k, v in sorted(class_indices.items(), key=lambda kv: kv[1])]
print("Class indices:", class_indices)
assert class_indices == {"female": 0, "male": 1}
for name, g in [("Train", train_generator), ("Validation", validation_generator), ("Test", test_generator)]:
    xb, yb = g[0]; print(f"{name:10s} batch {xb.shape}, labels {yb.shape}, batches/epoch {len(g)}, pixel range [{xb.min():.2f}, {xb.max():.2f}]")

In [ ]:
images, labels = train_generator[0]
plt.figure(figsize=(14, 3))
for i in range(6):
    plt.subplot(1, 6, i + 1); plt.imshow(np.clip(images[i], 0, 1)); plt.title(class_labels[int(labels[i])]); plt.axis("off")
plt.suptitle("Augmented training samples"); plt.show()

### 💡 Insight
The pixel range check confirms normalisation (0–1). The augmented samples should still look like clearly recognisable faces; if they are too distorted, reduce the augmentation strength.

## 4. Model Building
Four models are trained so the final choice is justified by evidence: **Baseline (Dense)** → **CNN v1** → **CNN v2 (BatchNorm + Dropout + L2)** → **MobileNetV2 transfer learning**.
All use `EarlyStopping` and `ReduceLROnPlateau` on `val_loss` with best-weights restore, and class weights.

In [ ]:
n_classes = 1                                   # single sigmoid unit = P(male)
counts = train_data["label"].value_counts()
class_weight = {class_indices[c]: len(train_data) / (2 * n) for c, n in counts.items()}
print("class_weight:", class_weight)

def get_metrics():                              # fresh metric objects for every model (never share them)
    return ["accuracy", keras.metrics.AUC(name="auc")]

def get_callbacks(patience):
    return [EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)]

def plot_history(hist, title):
    ep = range(1, len(hist["accuracy"]) + 1)
    plt.figure(figsize=(12, 4.5))
    plt.subplot(1, 2, 1); plt.plot(ep, hist["accuracy"], label="train"); plt.plot(ep, hist["val_accuracy"], label="validation")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title(f"Accuracy - {title}"); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(ep, hist["loss"], label="train"); plt.plot(ep, hist["val_loss"], label="validation")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(f"Loss - {title}"); plt.legend()
    plt.tight_layout(); plt.show()
    b = int(np.argmin(hist["val_loss"]))
    print(f"Lowest val_loss {hist['val_loss'][b]:.4f} at epoch {b + 1} (val_accuracy there: {hist['val_accuracy'][b]:.4f})")

def build_baseline():
    m = models.Sequential([layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)), layers.Flatten(),
        layers.Dense(256, activation="relu"), layers.Dropout(0.3), layers.Dense(128, activation="relu"), layers.Dense(1, activation="sigmoid")], name="baseline_dense")
    m.compile(loss=binary_crossentropy, optimizer=Adam(1e-3), metrics=get_metrics()); return m

def build_cnn_v1():
    m = models.Sequential([layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
        layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(1, activation="sigmoid")], name="cnn_v1")
    m.compile(loss=binary_crossentropy, optimizer=Adam(1e-3), metrics=get_metrics()); return m

def conv_block(x, filters, drop):
    for _ in range(2):
        x = layers.Conv2D(filters, (3, 3), padding="same", use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    return layers.Dropout(drop)(x)

def build_cnn_v2():
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)); x = inp
    for f, d in [(32, 0.10), (64, 0.20), (128, 0.25), (256, 0.30)]:
        x = conv_block(x, f, d)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = models.Model(inp, out, name="cnn_v2")
    m.compile(loss=binary_crossentropy, optimizer=Adam(1e-3), metrics=get_metrics()); return m

USING_IMAGENET = True
def build_mobilenet():
    global USING_IMAGENET
    try:
        base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=PRETRAINED_WEIGHTS)
        USING_IMAGENET = PRETRAINED_WEIGHTS == "imagenet"
    except Exception as e:
        print("WARNING: could not load the ImageNet weights (", type(e).__name__, ") - using RANDOM weights; results will be much worse.")
        base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None); USING_IMAGENET = False
    base.trainable = False
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(2.0, offset=-1.0)(inp)               # [0,1] -> [-1,1] as MobileNetV2 expects
    x = base(x, training=False)                               # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = models.Model(inp, out, name="mobilenetv2_gender")
    m.compile(loss=binary_crossentropy, optimizer=Adam(1e-3), metrics=get_metrics()); return m, base

def fit(model, epochs, patience):
    return model.fit(train_generator, epochs=epochs, validation_data=validation_generator,
                     class_weight=class_weight, callbacks=get_callbacks(patience), verbose=1)

### 4.1 Baseline — feed-forward (Dense) network

In [ ]:
baseline_model = build_baseline(); baseline_model.summary()
baseline_history = fit(baseline_model, EPOCHS["baseline"], patience=5)
plot_history(baseline_history.history, "Baseline")

### 4.2 CNN v1 — plain convolutional network

In [ ]:
cnn_model = build_cnn_v1(); cnn_model.summary()
cnn_history = fit(cnn_model, EPOCHS["cnn1"], patience=6)
plot_history(cnn_history.history, "CNN v1")

### 4.3 CNN v2 — deeper, regularised
Two conv layers per block with `BatchNormalization`, progressive `Dropout`, `L2` weight decay and `GlobalAveragePooling` (far fewer dense parameters). BatchNorm + Dropout models usually need more epochs to start learning, so patience is higher.

In [ ]:
cnn_model_v2 = build_cnn_v2(); cnn_model_v2.summary()
cnn_v2_history = fit(cnn_model_v2, EPOCHS["cnn2"], patience=8)
plot_history(cnn_v2_history.history, "CNN v2")

### 4.4 CNN v3 — transfer learning with MobileNetV2
A backbone pretrained on ImageNet already knows edges, textures and facial parts. Two phases:
1. **Feature extraction:** backbone frozen, only the new head trained (`lr = 1e-3`).
2. **Fine-tuning:** the top ~40 backbone layers unfrozen at a tiny learning rate (`lr = 1e-5`); BatchNorm layers stay frozen.

> Needs internet on the first run to download the ImageNet weights. If the download fails the notebook prints a warning (results will be worse with random weights).

In [ ]:
tl_model, base = build_mobilenet(); tl_model.summary()
print("Pretrained ImageNet weights in use:", USING_IMAGENET)
tl_hist1 = fit(tl_model, EPOCHS["tl1"], patience=4)                       # phase 1: head only

In [ ]:
# Phase 2: fine-tune the top of the backbone with a very small learning rate
base.trainable = True
for layer in base.layers[:-40]: layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization): layer.trainable = False
tl_model.compile(loss=binary_crossentropy, optimizer=Adam(1e-5), metrics=get_metrics())
tl_hist2 = fit(tl_model, EPOCHS["tl2"], patience=6)

tl_history = {k: tl_hist1.history[k] + tl_hist2.history[k] for k in tl_hist1.history if k in tl_hist2.history}
plot_history(tl_history, "CNN v3 (MobileNetV2, 2 phases)")

## 5. Model Comparison
Every model is scored on **train (un-augmented), validation and test** data at its restored best weights. The final model is chosen by **lowest validation loss** — the test set is not used for selection.

In [ ]:
models_dict = {"Baseline (Dense)": baseline_model, "CNN v1": cnn_model,
               "CNN v2 (BN + Dropout + L2)": cnn_model_v2, "CNN v3 (MobileNetV2)": tl_model}
rows = []
for name, m in models_dict.items():
    tr_loss, tr_acc, _ = m.evaluate(train_eval_generator, verbose=0)
    va_loss, va_acc, va_auc = m.evaluate(validation_generator, verbose=0)
    te_loss, te_acc, te_auc = m.evaluate(test_generator, verbose=0)
    rows.append([name, m.count_params(), tr_acc, va_acc, te_acc, va_loss, te_loss, te_auc])
comparison_df = pd.DataFrame(rows, columns=["Model", "Params", "Train Acc", "Val Acc", "Test Acc", "Val Loss", "Test Loss", "Test AUC"])
display(comparison_df.round(4))

best_idx = int(comparison_df["Val Loss"].idxmin())
best_model_name = comparison_df.loc[best_idx, "Model"]
final_model = list(models_dict.values())[best_idx]
print("Best model (lowest validation loss):", best_model_name)

plt.figure(figsize=(11, 5)); x = np.arange(len(comparison_df)); w_ = 0.27
for k, (col, lab) in enumerate([("Train Acc", "Train"), ("Val Acc", "Validation"), ("Test Acc", "Test")]):
    plt.bar(x + (k - 1) * w_, comparison_df[col], w_, label=lab)
plt.xticks(x, comparison_df["Model"], rotation=15); plt.ylabel("Accuracy"); plt.ylim(0.4, 1.0); plt.legend()
plt.title("Train vs validation vs test accuracy"); plt.tight_layout(); plt.show()

### 💡 Insight
- **Train ≫ validation/test** accuracy signals overfitting; **all three close** (and high) is the goal.
- Expect the CNNs to beat the Dense baseline (pixels flattened lose spatial structure) and the pretrained MobileNetV2 to give the best accuracy/lowest loss — but read the table, not the expectation: the selection is automatic.
- The test set contains a few hundred images, so differences of 1–2 points between models are within noise (see the confidence interval below).

## 6. Error Analysis of the Final Model — confusion matrix, report, ROC, misclassified images

In [ ]:
test_generator.reset()
y_prob = final_model.predict(test_generator, verbose=0).ravel()          # P(male)
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_generator.classes
acc = (y_pred == y_true).mean(); n = len(y_true)
z = 1.96; centre = (acc + z**2 / (2 * n)) / (1 + z**2 / n); half = z * np.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
print(f"Final model: {best_model_name}")
print(f"Test accuracy = {acc:.4f} on {n} images  (95% confidence interval: {centre - half:.3f} - {centre + half:.3f})")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels, ax=ax[0])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True"); ax[0].set_title("Confusion matrix - test set")
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, y_prob):.4f}"); ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("ROC curve"); ax[1].legend()
plt.tight_layout(); plt.show()
print(classification_report(y_true, y_pred, target_names=class_labels, digits=3))

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} misclassified out of {len(y_true)}")
if len(wrong):
    plt.figure(figsize=(15, 3.6))
    for i, idx in enumerate(wrong[:6]):
        plt.subplot(1, 6, i + 1); plt.imshow(Image.open(test_data["image_path"][idx]).convert("RGB")); plt.axis("off")
        conf = y_prob[idx] if y_pred[idx] == 1 else 1 - y_prob[idx]
        plt.title(f"true: {class_labels[y_true[idx]]}\npred: {class_labels[y_pred[idx]]} ({conf:.2f})", fontsize=9)
    plt.tight_layout(); plt.show()

### 💡 Insight
Check whether the errors are **balanced** between the two classes (confusion matrix) or concentrated in one. The misclassified images usually show the limits of the data: very small or blurry faces, heavy occlusion, extreme lighting or pose. These cases show what to improve next (e.g. detect and crop the face first).

## 7. Prediction on a new, unseen image

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_gender(img_path, model=None):
    # Same preprocessing as training: resize to IMG_SIZE, scale to 0-1.
    model = model or final_model
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(image.img_to_array(img) / 255.0, axis=0)
    p_male = float(model.predict(arr, verbose=0)[0][0])
    idx = int(p_male >= 0.5)
    return class_labels[idx], (p_male if idx == 1 else 1 - p_male)

img_path = test_data["image_path"][0]        # replace with your own photo, e.g. "my_photo.jpg"
label, conf = predict_gender(img_path)
print(f"Predicted gender: {label}  (confidence {conf:.1%})  |  true label: {test_data['label'][0]}")
plt.imshow(Image.open(img_path).convert("RGB")); plt.axis("off"); plt.title(f"{label} ({conf:.1%})"); plt.show()

## 8. Save the final model and verify it reloads correctly

In [ ]:
final_model.save("gender_detection_final.keras")
comparison_df.round(4).to_csv("model_comparison.csv", index=False)
json.dump({"model": best_model_name, "img_size": IMG_SIZE, "class_indices": class_indices, "rescale": "pixels / 255 (0-1)",
           "output": "sigmoid = P(male); threshold 0.5"}, open("gender_detection_meta.json", "w"), indent=2)

reloaded = keras.models.load_model("gender_detection_final.keras")
xb, _ = test_generator[0]
same = np.allclose(final_model.predict(xb, verbose=0), reloaded.predict(xb, verbose=0), atol=1e-5)
print("Saved gender_detection_final.keras | reloaded model gives identical predictions:", same)

## 9. Conclusion
**What was done** — audited the data (balance, corrupt files, sizes, duplicates), preprocessed it (resize, normalise, training-only augmentation), trained and compared four models (Dense baseline, CNN v1, regularised CNN v2, MobileNetV2 transfer learning), selected the final model by **validation loss**, analysed its errors on the untouched test set and saved a verified model.

**How to read the results** — the comparison table (section 5) and the test accuracy with its confidence interval (section 6) are the final numbers of *your* run; the notebook does not hard-code any result.

**Limitations**
- The model judges **appearance only**, can inherit dataset biases (age, ethnicity, lighting, background), and must not be used for high-stakes decisions.
- Full images are used as-is (no face detection/cropping), so background adds noise.
- One hold-out split of a few hundred images gives a noisy estimate; k-fold cross-validation would be more robust.

**Next steps**
- Detect and crop the face first (OpenCV Haar cascade / MTCNN / RetinaFace).
- Try EfficientNetB0 / ResNet50 backbones, test-time augmentation (original + flipped image), and a hyper-parameter search (Keras Tuner).
- Evaluate accuracy per sub-group (age, lighting, pose) to check fairness before any deployment.